# 11 - Cluster Profiling and Default Analysis
### Credit Portfolio Segmentation and Risk Pattern Analysis

---

## 📌 Beginner Objective
This is the most exciting milestone of the project!
We now bring back the **`Default`** column and evaluate our segments:
1. Did our unsupervised clusters naturally separate good borrowers from bad borrowers?
2. What are the average income, FICO score, and loan amount for each segment?
3. What are the historical default rates across segments?
4. Assign meaningful, professional **Business Personas** to each segment.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

# Load data and scaled features
df = pd.read_csv("../data/processed/cleaned_loans_dataset.csv")
X_scaled = pd.read_csv("../data/processed/scaled_features.csv")

# Fit final K-Means (k=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

print("Clusters assigned successfully!")


---
## 1. Calculate Cluster Statistics & Default Rates


In [ ]:
profile = df.groupby('KMeans_Cluster').agg(
    Borrower_Count=('Default', 'count'),
    Avg_Income=('annual_income', 'mean'),
    Avg_FICO=('fico_score', 'mean'),
    Avg_DTI=('dti', 'mean'),
    Avg_Loan_Amount=('loan_amount', 'mean'),
    Default_Rate_Pct=('Default', lambda x: round(x.mean() * 100, 2))
).reset_index()

profile.round(2)


---
## 2. Assign Human-Readable Segment Personas
Looking at the numbers above, we map each cluster to a business persona:


In [ ]:
# Naming clusters based on their financial properties
persona_map = {
    0: "Prime / Low Risk",
    1: "Overleveraged / Moderate Risk",
    2: "Entry-Level / Young Profile",
    3: "Subprime / High Risk"
}

df['Segment_Name'] = df['KMeans_Cluster'].map(persona_map)
profile['Segment_Name'] = profile['KMeans_Cluster'].map(persona_map)

# Save summary table
profile.to_csv("../outputs/tables/cluster_risk_profiles.csv", index=False)
profile[['Segment_Name', 'Borrower_Count', 'Avg_Income', 'Avg_FICO', 'Default_Rate_Pct']]


---
## 3. Visualize Default Rates by Segment


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(x='Segment_Name', y='Default_Rate_Pct', data=profile, palette='magma')
plt.title("Historical Loan Default Rate by Borrower Segment (%)")
plt.xlabel("Borrower Segment")
plt.ylabel("Default Rate (%)")
plt.xticks(rotation=15)

# Annotate bars with exact percentages
for idx, row in profile.iterrows():
    plt.text(idx, row['Default_Rate_Pct'] + 0.5, f"{row['Default_Rate_Pct']}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig("../outputs/figures/cluster_default_rates.png", dpi=300)
plt.show()

# Save final segmented dataset
df.to_csv("../data/processed/segmented_loans_dataset.csv", index=False)
print("Saved final segmented data to ../data/processed/segmented_loans_dataset.csv!")
